# AeroPulse Propagation — 05: Hybrid Residual Models

Per-horizon residual-over-baseline models, compared across gradient-boosting
families, tuned on validation, and ablated to find out which feature blocks
actually earn their place.

## Objective

Improvement plan sections 17, 18, 29 and 52 (Notebook 03 changes).

The residual architecture is kept: the model predicts the *change* from a
known baseline, not the concentration. What changes:

1. the baseline is the **best baseline per horizon** from notebook 04, not
   persistence by default — a residual model built on a weak baseline
   inherits its weakness
2. **four model families** compared under one contract (LightGBM, XGBoost,
   CatBoost, HistGradientBoosting) rather than one assumed winner
3. **Optuna tuning on validation only**; the test period stays sealed
4. a **feature-block ablation**, so the claim "transport features help" is
   measured rather than asserted
5. per-horizon **feature importance** exported for the inference layer

The test split is scored once, at the end.

In [1]:
# ============================================================================
# SETUP — everything reusable lives in propagation_toolkit.py, not in cells.
# ============================================================================
import os, sys, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import propagation_toolkit as pt

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

cfg = pt.load_config()
HORIZONS = cfg.horizons
print("kernel:", sys.executable)

env: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/propagation_forecast/.env
project: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/propagation_forecast
horizons: [1, 3, 6, 12, 24, 48] | schema 2.0.0
kernel: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/.venv/bin/python


In [2]:
from sklearn.model_selection import ParameterSampler
import joblib

MAX_TRAIN_ROWS = cfg.max_train_rows or 500_000   # PROP_MAX_TRAIN_ROWS overrides
TUNING_ROWS = 200_000
N_TUNING_TRIALS = int(os.getenv("PROP_TUNING_TRIALS", "20"))
print("train cap", f"{MAX_TRAIN_ROWS:,}", "| tuning rows", f"{TUNING_ROWS:,}",
      "| tuning trials", N_TUNING_TRIALS)

feature_file = cfg.feature_dir / "propagation_features.parquet"
df = pd.read_parquet(feature_file)
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)
df = df.sort_values(["location_id", "timestamp_utc"]).reset_index(drop=True)
print("rows", f"{len(df):,}", "| columns", df.shape[1])

bar = pd.read_csv(cfg.stage_dir("baselines") / "baseline_bar.csv")
BEST_BASELINE = dict(zip(bar["horizon_h"], bar["best_baseline"]))
print("baseline per horizon:", BEST_BASELINE)

train cap 500,000 | tuning rows 200,000 | tuning trials 20


rows 1,705,252 | columns 306
baseline per horizon: {1: 'persistence', 3: 'ensemble', 6: 'ensemble', 12: 'ensemble', 24: 'ensemble', 48: 'ensemble'}


## 1. Build the modelling frame for one horizon

`prepare(h)` attaches the baseline ladder, picks the horizon's baseline as the
residual anchor, and returns the chronological split. Nothing here touches
the test rows beyond selecting them.

In [3]:
def subsample(frame, cap, seed=42):
    """Even thinning across time, not a random draw of recent rows."""
    if cap is None or len(frame) <= cap:
        return frame
    step = len(frame) / cap
    idx = (np.arange(cap) * step).astype(int)
    return frame.iloc[idx]


def prepare(h, cap=MAX_TRAIN_ROWS):
    target_col = f"target_pm25_t_plus_{h}h"
    work = df.dropna(subset=[target_col, "pm25"]).copy()
    baselines = pt.baseline_forecasts(work, h)
    for name in pt.BASELINE_NAMES:
        work[f"bl_{name}"] = baselines[name]
    anchor = BEST_BASELINE[h]
    work["baseline_forecast"] = work[f"bl_{anchor}"].fillna(work["pm25"])
    train, valid, test = pt.chrono_split(work, h, cfg.train_fraction, cfg.validation_fraction)
    return subsample(train, cap), valid, test, target_col, anchor


for h in HORIZONS:
    train, valid, test, target_col, anchor = prepare(h, cap=None)
    print(f"h={h:>2}  anchor={anchor:<12} train {len(train):>9,}  valid {len(valid):>8,}  test {len(test):>8,}")

h= 1  anchor=persistence  train 1,153,817  valid  247,112  test  247,308


h= 3  anchor=ensemble     train 1,127,794  valid  241,407  test  241,861


h= 6  anchor=ensemble     train 1,108,239  valid  236,809  test  237,773


h=12  anchor=ensemble     train 1,085,275  valid  231,426  test  232,962


h=24  anchor=ensemble     train 1,070,761  valid  227,130  test  230,216


h=48  anchor=ensemble     train 1,048,745  valid  221,013  test  226,059


## 2. Model family comparison

Same features, same split, same residual target. Trained on a capped
subsample so the comparison finishes in reasonable time; the winner is
retrained on the full training window in section 5.

In [4]:
FAMILIES = ["lightgbm", "xgboost", "catboost", "hist_gbr"]
COMPARISON_HORIZONS = [h for h in (6, 24) if h in HORIZONS] or HORIZONS[:2]
COMPARISON_CAP = 250_000

rows = []
for h in COMPARISON_HORIZONS:
    train, valid, _, target_col, anchor = prepare(h, cap=COMPARISON_CAP)
    features = pt.select_features(train, horizon=h)
    for kind in FAMILIES:
        t0 = time.time()
        model = pt.ResidualModel.fit(train, features, h, kind=kind)
        fit_s = time.time() - t0
        t0 = time.time()
        pred = model.predict(valid)
        predict_ms = (time.time() - t0) / len(valid) * 1000
        metrics = pt.regression_metrics(
            valid[target_col].to_numpy(dtype=float), pred,
            valid["baseline_forecast"].to_numpy(dtype=float),
        )
        rows.append({
            "horizon_h": h, "family": kind, "n_features": len(features),
            "valid_mae": round(metrics["mae"], 3),
            "baseline_mae": round(metrics["mae_reference"], 3),
            "skill_vs_baseline": round(metrics["skill"], 4),
            "fit_seconds": round(fit_s, 1),
            "predict_ms_per_row": round(predict_ms, 5),
        })
        print(f"  h={h:>2} {kind:<12} valid MAE {metrics['mae']:.3f}  "
              f"skill {metrics['skill']:+.4f}  fit {fit_s:.0f}s")

families = pd.DataFrame(rows)
print()
print(families.to_string(index=False))

  h= 6 lightgbm     valid MAE 10.179  skill +0.1079  fit 13s


  h= 6 xgboost      valid MAE 10.314  skill +0.0961  fit 13s


  h= 6 catboost     valid MAE 10.209  skill +0.1052  fit 14s


  h= 6 hist_gbr     valid MAE 10.241  skill +0.1024  fit 16s


  h=24 lightgbm     valid MAE 11.647  skill +0.0628  fit 12s


  h=24 xgboost      valid MAE 11.865  skill +0.0452  fit 12s


  h=24 catboost     valid MAE 11.364  skill +0.0855  fit 14s


  h=24 hist_gbr     valid MAE 11.562  skill +0.0696  fit 15s

 horizon_h   family  n_features  valid_mae  baseline_mae  skill_vs_baseline  fit_seconds  predict_ms_per_row
         6 lightgbm         225     10.179        11.410             0.1079         12.7             0.00344
         6  xgboost         225     10.314        11.410             0.0961         12.8             0.00083
         6 catboost         225     10.209        11.410             0.1052         13.7             0.00029
         6 hist_gbr         225     10.241        11.410             0.1024         15.6             0.00235
        24 lightgbm         224     11.647        12.427             0.0628         12.4             0.00327
        24  xgboost         224     11.865        12.427             0.0452         11.9             0.00070
        24 catboost         224     11.364        12.427             0.0855         13.7             0.00028
        24 hist_gbr         224     11.562        12.427          

In [5]:
ranking = (families.groupby("family")
           .agg(mean_skill=("skill_vs_baseline", "mean"),
                mean_mae=("valid_mae", "mean"),
                mean_fit_s=("fit_seconds", "mean"))
           .sort_values("mean_skill", ascending=False).round(4))
print(ranking.to_string())
WINNER = ranking.index[0]
print("\nselected family:", WINNER)

          mean_skill  mean_mae  mean_fit_s
family                                    
catboost      0.0954   10.7865       13.70
hist_gbr      0.0860   10.9015       15.20
lightgbm      0.0854   10.9130       12.55
xgboost       0.0706   11.0895       12.35

selected family: catboost


## 3. Hyperparameter tuning on validation

Random search over the plan's section 29 grid. The test period is not read.

In [6]:
# One canonical search space. pt.translate_params renames the keys per family
# (CatBoost wants `depth`, sklearn wants `max_iter`), so the winner of the
# comparison above is tuned over exactly the same grid as any other family.
SEARCH_SPACE = {
    "learning_rate": [0.02, 0.03, 0.05, 0.08, 0.12],
    "num_leaves": [31, 63, 127, 255],
    "max_depth": [-1, 6, 8, 12],
    "min_child_samples": [20, 40, 80, 150],
    "subsample": [0.6, 0.7, 0.8, 0.9],
    "colsample_bytree": [0.5, 0.6, 0.8, 1.0],
    "reg_alpha": [0.0, 0.1, 1.0],
    "reg_lambda": [0.0, 1.0, 5.0],
    "n_estimators": [300, 500, 800],
}
TUNE_HORIZON = 6 if 6 in HORIZONS else HORIZONS[len(HORIZONS) // 2]
train, valid, _, target_col, anchor = prepare(TUNE_HORIZON, cap=TUNING_ROWS)
features = pt.select_features(train, horizon=TUNE_HORIZON)
valid_sample = subsample(valid, 120_000)

trials = []
t0 = time.time()
for i, params in enumerate(ParameterSampler(SEARCH_SPACE, n_iter=N_TUNING_TRIALS, random_state=42)):
    # pt.translate_params maps this canonical dict onto whichever library won,
    # so the search space is the same regardless of family.
    model = pt.ResidualModel.fit(train, features, TUNE_HORIZON, kind=WINNER, params=params)
    mae = float(np.abs(model.predict(valid_sample) - valid_sample[target_col]).mean())
    trials.append({**params, "valid_mae": round(mae, 4)})
    if i % 5 == 0:
        print(f"  trial {i:>2}  MAE {mae:.4f}")

tuning = pd.DataFrame(trials).sort_values("valid_mae")
print(f"\n{N_TUNING_TRIALS} trials in {time.time() - t0:.0f}s")
print(tuning.head(5).to_string(index=False))

BEST_PARAMS = {k: v for k, v in tuning.iloc[0].items() if k != "valid_mae"}
BEST_PARAMS = {k: (int(v) if isinstance(v, float) and float(v).is_integer() else v)
               for k, v in BEST_PARAMS.items()}
default_mae = float(np.abs(
    pt.ResidualModel.fit(train, features, TUNE_HORIZON, kind=WINNER).predict(valid_sample)
    - valid_sample[target_col]).mean())
print(f"\ndefault params MAE {default_mae:.4f} -> tuned {tuning.iloc[0]['valid_mae']:.4f} "
      f"({1 - tuning.iloc[0]['valid_mae'] / default_mae:+.2%})")
print("best:", BEST_PARAMS)

  trial  0  MAE 10.3119


  trial  5  MAE 10.2251


  trial 10  MAE 10.3701


  trial 15  MAE 10.2620



20 trials in 640s
 subsample  reg_lambda  reg_alpha  num_leaves  n_estimators  min_child_samples  max_depth  learning_rate  colsample_bytree  valid_mae
       0.8         5.0        0.1          31           300                150         12           0.05               0.6    10.2251
       0.7         5.0        0.0         255           500                150          8           0.12               0.6    10.2336
       0.8         1.0        0.1         127           500                 80          8           0.05               0.8    10.2531
       0.7         1.0        0.1         255           300                 80          6           0.12               0.6    10.2578
       0.9         5.0        0.0         255           300                 80          8           0.08               0.6    10.2620



default params MAE 10.2767 -> tuned 10.2251 (+0.50%)
best: {'subsample': 0.8, 'reg_lambda': 5, 'reg_alpha': 0.1, 'num_leaves': 31, 'n_estimators': 300, 'min_child_samples': 150, 'max_depth': 12, 'learning_rate': 0.05, 'colsample_bytree': 0.6}


## 4. Feature-block ablation

Does each block the improvement plan asked for actually pay for itself?
Drop one block at a time and re-measure on validation. A block whose removal
does not hurt is a block carrying maintenance cost for nothing.

In [7]:
BLOCKS = {
    "transport (advect/fcst)": lambda c: c.startswith(("advect_", "transport_", "fcst_")),
    "station climatology":     lambda c: c in pt.FEATURE_COLUMNS_STATION,
    "stability / stagnation":  lambda c: c.startswith(("stagnation_", "ventilation_")),
    "regime flags":            lambda c: c.startswith("regime_"),
    "fire":                    lambda c: c.startswith(("fire_", "upwind_fire_", "crosswind_fire", "downwind_fire")),
    "spatial neighbour field": lambda c: c.startswith(("regional_", "upwind_pm25", "national_", "pm25_local_excess")),
    "CAMS satellite":          lambda c: c.startswith(("cams_", "satellite_")),
    "pm25 history (lags/rolls)": lambda c: c.startswith(("pm25_lag_", "pm25_roll_", "pm25_delta_", "pm25_slope_", "pm25_pct_", "pm25_zscore", "pm25_acceler", "pm25_percentile")),
}

ABLATION_HORIZON = TUNE_HORIZON
train, valid, _, target_col, anchor = prepare(ABLATION_HORIZON, cap=TUNING_ROWS)
full_features = pt.select_features(train, horizon=ABLATION_HORIZON)
valid_sample = subsample(valid, 120_000)

full_model = pt.ResidualModel.fit(train, full_features, ABLATION_HORIZON, kind=WINNER, params=BEST_PARAMS)
full_mae = float(np.abs(full_model.predict(valid_sample) - valid_sample[target_col]).mean())
print(f"all {len(full_features)} features: valid MAE {full_mae:.4f}\n")

rows = []
for label, predicate in BLOCKS.items():
    kept = [c for c in full_features if not predicate(c)]
    dropped = len(full_features) - len(kept)
    if dropped == 0:
        continue
    model = pt.ResidualModel.fit(train, kept, ABLATION_HORIZON, kind=WINNER, params=BEST_PARAMS)
    mae = float(np.abs(model.predict(valid_sample) - valid_sample[target_col]).mean())
    rows.append({
        "block_removed": label,
        "features_dropped": dropped,
        "valid_mae": round(mae, 4),
        "mae_increase": round(mae - full_mae, 4),
        "pct_worse": round((mae / full_mae - 1) * 100, 3),
    })
    print(f"  -{label:<28} ({dropped:>3} feats)  MAE {mae:.4f}  {(mae / full_mae - 1) * 100:+.2f}%")

ablation = pd.DataFrame(rows).sort_values("mae_increase", ascending=False)
print()
print(ablation.to_string(index=False))

all 225 features: valid MAE 10.2251



  -transport (advect/fcst)      ( 12 feats)  MAE 10.2498  +0.24%


  -station climatology          ( 11 feats)  MAE 10.2053  -0.19%


  -stability / stagnation       ( 11 feats)  MAE 10.2500  +0.24%


  -regime flags                 (  6 feats)  MAE 10.2553  +0.30%


  -fire                         ( 46 feats)  MAE 10.2283  +0.03%


  -spatial neighbour field      ( 15 feats)  MAE 10.2584  +0.33%


  -CAMS satellite               ( 20 feats)  MAE 10.2386  +0.13%


  -pm25 history (lags/rolls)    ( 40 feats)  MAE 10.3484  +1.21%

            block_removed  features_dropped  valid_mae  mae_increase  pct_worse
pm25 history (lags/rolls)                40    10.3484        0.1233      1.206
  spatial neighbour field                15    10.2584        0.0333      0.325
             regime flags                 6    10.2553        0.0302      0.295
   stability / stagnation                11    10.2500        0.0249      0.243
  transport (advect/fcst)                12    10.2498        0.0246      0.241
           CAMS satellite                20    10.2386        0.0135      0.132
                     fire                46    10.2283        0.0032      0.031
      station climatology                11    10.2053       -0.0198     -0.194


## 5. Train the final per-horizon models

In [8]:
models = {}
rows = []
for h in HORIZONS:
    train, valid, test, target_col, anchor = prepare(h)
    features = pt.select_features(train, horizon=h)
    t0 = time.time()
    model = pt.ResidualModel.fit(train, features, h, kind=WINNER, params=BEST_PARAMS)
    fit_s = time.time() - t0

    record = {"horizon_h": h, "anchor_baseline": anchor, "n_features": len(features),
              "n_train": len(train), "fit_seconds": round(fit_s, 1)}
    for split_name, part in (("valid", valid), ("test", test)):
        pred = model.predict(part)
        y = part[target_col].to_numpy(dtype=float)
        anchor_pred = part["baseline_forecast"].to_numpy(dtype=float)
        persistence = part["pm25"].to_numpy(dtype=float)
        metrics = pt.regression_metrics(y, pred, anchor_pred, prefix=f"{split_name}_")
        record.update({k: (round(v, 4) if isinstance(v, float) else v) for k, v in metrics.items()})
        record[f"{split_name}_skill_vs_persistence"] = round(
            pt.skill(metrics[f"{split_name}_mae"], float(np.abs(persistence - y).mean())), 4)
    models[h] = model
    rows.append(record)
    print(f"h={h:>2} [{anchor:<12}] test MAE {record['test_mae']:.3f}  "
          f"vs baseline {record['test_mae_reference']:.3f}  "
          f"skill_baseline {record['test_skill']:+.4f}  "
          f"skill_persist {record['test_skill_vs_persistence']:+.4f}")

horizon_table = pd.DataFrame(rows)

h= 1 [persistence ] test MAE 4.047  vs baseline 4.201  skill_baseline +0.0365  skill_persist +0.0365


h= 3 [ensemble    ] test MAE 6.136  vs baseline 6.572  skill_baseline +0.0662  skill_persist +0.1292


h= 6 [ensemble    ] test MAE 6.810  vs baseline 7.313  skill_baseline +0.0688  skill_persist +0.1861


h=12 [ensemble    ] test MAE 7.316  vs baseline 7.968  skill_baseline +0.0818  skill_persist +0.2276


h=24 [ensemble    ] test MAE 7.638  vs baseline 8.203  skill_baseline +0.0689  skill_persist +0.1036


h=48 [ensemble    ] test MAE 8.608  vs baseline 9.127  skill_baseline +0.0569  skill_persist +0.1017


In [9]:
view = horizon_table[[
    "horizon_h", "anchor_baseline", "n_train", "valid_mae", "test_mae",
    "test_mae_reference", "test_skill", "test_skill_vs_persistence",
    "test_rmse", "test_r2", "test_bias",
]]
print(view.to_string(index=False))
print("\npositive skill vs its own baseline:",
      horizon_table.loc[horizon_table["test_skill"] > 0, "horizon_h"].tolist())
print("positive skill vs persistence only:",
      horizon_table.loc[horizon_table["test_skill_vs_persistence"] > 0, "horizon_h"].tolist())

 horizon_h anchor_baseline  n_train  valid_mae  test_mae  test_mae_reference  test_skill  test_skill_vs_persistence  test_rmse  test_r2  test_bias
         1     persistence   500000     5.2293    4.0474              4.2008      0.0365                     0.0365    12.1782   0.6095     0.0407
         3        ensemble   500000     8.7925    6.1363              6.5716      0.0662                     0.1292    15.5169   0.3678    -0.0409
         6        ensemble   500000    10.1503    6.8099              7.3133      0.0688                     0.1861    16.3227   0.3052    -0.0766
        12        ensemble   500000    11.1028    7.3158              7.9676      0.0818                     0.2276    17.2008   0.2548    -0.1576
        24        ensemble   500000    11.3813    7.6380              8.2029      0.0689                     0.1036    17.1245   0.2353     0.0128
        48        ensemble   500000    12.8311    8.6080              9.1274      0.0569                     0.1017   

## 6. Feature importance

In [10]:
importances = {}
for h in HORIZONS:
    imp = models[h].feature_importance()
    importances[h] = imp
    top = imp.head(8)
    print(f"\nh={h}  top features ({top['importance_pct'].sum():.1f}% of total gain):")
    for _, r in top.iterrows():
        print(f"    {r['feature']:<32} {r['importance_pct']:5.2f}%")


h=1  top features (31.3% of total gain):
    pm25_delta_1h                     5.77%
    pm25_lag_2h                       5.56%
    pm25_slope_24h                    5.26%
    pm25_pct_change_1h                5.17%
    bl_ensemble                       2.67%
    advect_lat_h1                     2.40%
    pm25_roll_std_6h                  2.33%
    pm25_vs_station_month             2.18%

h=3  top features (31.4% of total gain):
    bl_seasonal_168h                  4.77%
    sin_hour                          4.23%
    bl_ensemble                       4.12%
    delta_pressure_3h                 3.93%
    bl_seasonal_24h                   3.82%
    baseline_forecast                 3.57%
    cos_hour                          3.52%
    pm25_delta_24h                    3.45%

h=6  top features (30.1% of total gain):
    delta_pressure_3h                 4.36%
    bl_ensemble                       4.20%
    pm25_vs_station_month             4.06%
    baseline_forecast                 

## 7. Save

In [11]:
stage = cfg.stage_dir("hybrid")
families.to_csv(stage / "family_comparison.csv", index=False)
tuning.to_csv(stage / "tuning_trials.csv", index=False)
ablation.to_csv(stage / "feature_ablation.csv", index=False)
horizon_table.to_csv(stage / "horizon_metrics.csv", index=False)
for h, imp in importances.items():
    imp.to_csv(stage / f"feature_importance_{h}h.csv", index=False)
joblib.dump(models, stage / "residual_models.joblib")

report = {
    "generated_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "winner_family": WINNER,
    "best_params": BEST_PARAMS,
    "tuned_on_horizon": TUNE_HORIZON,
    "tuning_trials": N_TUNING_TRIALS,
    "train_cap": MAX_TRAIN_ROWS,
    "anchor_baselines": {str(k): v for k, v in BEST_BASELINE.items()},
    "family_ranking": ranking.reset_index().to_dict(orient="records"),
    "ablation": ablation.to_dict(orient="records"),
    "horizon_metrics": horizon_table.to_dict(orient="records"),
}
(stage / "hybrid_report.json").write_text(json.dumps(report, indent=2, default=str))

print("05 COMPLETE ->", stage)
print("Next: 06_propagation_uncertainty.ipynb")

05 COMPLETE -> /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/propagation_forecast/artifacts/propagation/hybrid
Next: 06_propagation_uncertainty.ipynb


## Findings

**Every horizon beats its own baseline, and the baseline is the hard one.**

| horizon | anchor | test MAE | baseline MAE | skill vs baseline | skill vs persistence |
|---|---|---|---|---|---|
| 1 h | persistence | 4.05 | 4.20 | +0.037 | +0.037 |
| 3 h | ensemble | 6.14 | 6.57 | +0.066 | +0.129 |
| 6 h | ensemble | 6.81 | 7.31 | +0.069 | +0.186 |
| 12 h | ensemble | 7.32 | 7.97 | +0.082 | +0.228 |
| 24 h | ensemble | 7.64 | 8.20 | +0.069 | +0.104 |
| 48 h | ensemble | 8.61 | 9.13 | +0.057 | +0.102 |

Note the two skill columns. Against persistence the model looks strong
(+0.23 at 12 h); against the baseline it actually has to beat, it is +0.08.
**Roughly two thirds of the apparent skill at 12 h is available for free from
a weighted average of lagged observations.** That is the number to quote to
anyone deciding whether the ML model is worth operating.

R² falls from 0.61 at 1 h to 0.16 at 48 h, which is the honest shape of this
problem: past 24 h there is not much of the future left in the present.

**CatBoost wins, but the margin is inside the noise floor.** Mean validation
skill: CatBoost 0.0954, HistGradientBoosting 0.0860, LightGBM 0.0854,
XGBoost 0.0706, at essentially identical fit times (12-15 s). Only XGBoost is
clearly behind. CatBoost was selected because it won, not because the
difference is convincing — LightGBM would be a defensible swap if inference
cost ever favours it.

**Tuning bought 0.50%.** 20 random-search trials over the section 29 grid
moved validation MAE from 10.277 to 10.225. Worth doing once and not worth
doing often. The winning configuration is conservative (`num_leaves` 31,
`min_child_samples` 150, `reg_lambda` 5), which says the model is
variance-limited rather than capacity-limited — more trees or deeper trees
are not the lever.

**The ablation is the uncomfortable part.** Dropping each block and
re-measuring:

| block removed | features | MAE change |
|---|---|---|
| pm25 history (lags/rolls) | 40 | **+1.21%** |
| spatial neighbour field | 15 | +0.33% |
| regime flags | 6 | +0.30% |
| stability / stagnation | 11 | +0.24% |
| transport (advect/fcst) | 12 | +0.24% |
| CAMS satellite | 20 | +0.13% |
| fire | 46 | +0.03% |
| station climatology | 11 | **-0.19%** |

Only PM2.5's own history matters, by roughly 4x over anything else. The 46
fire columns are worth 0.03%. The transport features that notebook 03 spent
real effort correcting are worth 0.24% — the unit fix and the integration
were the right things to do, but they are not what makes this model work.

And the station climatology block, added in notebook 02 specifically because
the plan's section 10 asked for it, makes validation MAE **slightly worse**.

That last result sits directly against the feature importances, where
`pm25_vs_station_month` and `station_baseline_by_month` rank in the top five
at 12 h, 24 h and 48 h. Both readings are correct and they measure different
things: gain measures how often the model *reaches for* a feature, ablation
measures whether it *needs* it. The station features are heavily used and
almost perfectly substitutable — with them gone, `bl_seasonal_168h` and the
calendar columns carry the same signal. The recommendation is to keep them
(they cost nothing at inference and they make the drivers legible in the UI)
while being clear that they are not earning skill.

**What the models actually key on shifts with horizon.** At 1 h it is pure
short-term dynamics (`pm25_delta_1h`, `pm25_lag_2h`, `pm25_pct_change_1h`).
By 24-48 h the top features are `bl_seasonal_168h`, `pm25_vs_station_month`,
`station_baseline_by_month` and the Diwali calendar distance — the model has
stopped forecasting and started interpolating climatology, which is the
correct thing for it to do and a fair warning about what a 48 h number means.